# Hybrid retrieval & re-ranking
Hybrid retrieval joins sparse precision with dense semantic recall, then re-ranking spends extra compute only on the shortlist. BM25 contributes exact lexical evidence while dense retrieval contributes semantic evidence. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 16
rng = np.random.default_rng(SEED)
random.seed(SEED)


def l2_normalize(x):
    x = np.asarray(x, dtype=float)
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norm, 1e-12)


def minmax(x):
    x = np.asarray(x, dtype=float)
    lo = np.min(x)
    hi = np.max(x)
    span = max(float(hi - lo), 1e-12)
    return (x - lo) / span


def cosine_scores(query, docs):
    q = l2_normalize(np.asarray(query, dtype=float).reshape(1, -1))[0]
    d = l2_normalize(np.asarray(docs, dtype=float))
    return d @ q


def euclidean_distances(points, query):
    points = np.asarray(points, dtype=float)
    query = np.asarray(query, dtype=float)
    return np.linalg.norm(points - query, axis=1)


def rank_desc(scores):
    scores = np.asarray(scores, dtype=float)
    return list(np.argsort(-scores, kind="mergesort"))


def rank_asc(scores):
    scores = np.asarray(scores, dtype=float)
    return list(np.argsort(scores, kind="mergesort"))


def recall_at_1(pred, truth):
    return float(int(pred == truth))


def dcg_at_k(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    discounts = np.log2(np.arange(2, len(rels) + 2))
    gains = np.power(2.0, rels) - 1.0
    return float(np.sum(gains / discounts))


def ndcg_at_k(rels, k):
    rels = list(rels)
    ideal = sorted(rels, reverse=True)
    denom = dcg_at_k(ideal, k)
    if denom == 0.0:
        return 0.0
    return dcg_at_k(rels, k) / denom


def average_precision(rels):
    total = 0.0
    hits = 0
    for index, rel in enumerate(rels, start=1):
        if rel > 0:
            hits = hits + 1
            total = total + hits / index
    if hits == 0:
        return 0.0
    return total / hits


def ann_candidates_then_exact(points, query, centroids, assignments, nprobe=1, graph=None, entry=0, ef=3):
    points = np.asarray(points, dtype=float)
    query = np.asarray(query, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    center_order = rank_asc(euclidean_distances(centroids, query))[:nprobe]
    ivf_candidates = []
    for center_id in center_order:
        members = np.where(np.asarray(assignments) == center_id)[0]
        ivf_candidates.extend(list(members))
    if graph is not None:
        seen = {entry}
        frontier = [entry]
        while frontier and len(seen) < ef:
            current = min(frontier, key=lambda i: np.linalg.norm(points[i] - query))
            frontier.remove(current)
            for neighbor in graph.get(current, []):
                if neighbor not in seen:
                    seen.add(neighbor)
                    frontier.append(neighbor)
        ivf_candidates.extend(sorted(seen))
    candidates = sorted(set(ivf_candidates))
    if not candidates:
        candidates = list(range(len(points)))
    local_distances = euclidean_distances(points[candidates], query)
    best_local = int(np.argmin(local_distances))
    best_index = int(candidates[best_local])
    return best_index, candidates, local_distances


def hybrid_fuse_and_rerank(bm25_scores, dense_scores, rerank_scores=None, alpha=0.5, lam=0.4, normalize=True, top_k=None):
    bm25_scores = np.asarray(bm25_scores, dtype=float)
    dense_scores = np.asarray(dense_scores, dtype=float)
    if normalize:
        sparse = minmax(bm25_scores)
        dense = minmax(dense_scores)
    else:
        sparse = bm25_scores
        dense = dense_scores
    hybrid = alpha * sparse + (1.0 - alpha) * dense
    order = rank_desc(hybrid)
    if top_k is not None:
        order = order[:top_k]
    if rerank_scores is None:
        return hybrid, order, hybrid
    rerank_scores = np.asarray(rerank_scores, dtype=float)
    final = hybrid.copy()
    for doc_id in order:
        final[doc_id] = lam * hybrid[doc_id] + (1.0 - lam) * rerank_scores[doc_id]
    final_order = sorted(order, key=lambda i: -final[i])
    return hybrid, final_order, final


def bi_retrieve_then_cross_rerank(query_vec, doc_vecs, cross_scores, cutoff=3):
    bi_scores = cosine_scores(query_vec, doc_vecs)
    bi_order = rank_desc(bi_scores)
    shortlist = bi_order[:cutoff]
    cross_scores = np.asarray(cross_scores, dtype=float)
    final_order = sorted(shortlist, key=lambda i: -cross_scores[i])
    return bi_scores, shortlist, final_order


def retrieval_metrics(rels, k=5):
    rels = np.asarray(rels, dtype=float)
    top = rels[:k]
    total_relevant = max(float(np.sum(rels > 0)), 1.0)
    precision = float(np.sum(top > 0) / k)
    recall = float(np.sum(top > 0) / total_relevant)
    ndcg = ndcg_at_k(rels, k)
    ap = average_precision(rels)
    return {
        "precision": precision,
        "recall": recall,
        "ndcg": ndcg,
        "ap": ap,
    }


def ann_ladder():
    d1_points = np.array([
        [0.0, 0.0],
        [1.0, 0.0],
        [0.0, 1.0],
        [3.0, 3.0],
        [3.2, 3.1],
        [4.0, 3.0],
    ])
    d1_query = np.array([3.1, 3.0])
    d1_centroids = np.array([
        [0.33, 0.33],
        [3.4, 3.03],
    ])
    d1_assignments = np.array([0, 0, 0, 1, 1, 1])
    d2_points = np.vstack([
        rng.normal([0.0, 0.0], 0.10, size=(8, 2)),
        rng.normal([2.5, 2.5], 0.10, size=(8, 2)),
        rng.normal([5.0, 0.0], 0.10, size=(8, 2)),
    ])
    d2_query = np.array([2.46, 2.52])
    d2_centroids = np.array([
        [0.0, 0.0],
        [2.5, 2.5],
        [5.0, 0.0],
    ])
    d2_assignments = np.repeat(np.arange(3), 8)
    d3_points = np.vstack([
        rng.normal([0.0, 0.0], 0.20, size=(10, 2)),
        rng.normal([1.0, 1.0], 0.25, size=(10, 2)),
        rng.normal([1.4, 1.2], 0.25, size=(10, 2)),
    ])
    d3_query = np.array([1.22, 1.08])
    d3_centroids = np.array([
        [0.0, 0.0],
        [1.0, 1.0],
        [1.4, 1.2],
    ])
    d3_assignments = np.repeat(np.arange(3), 10)
    topic_centers = np.array([
        [1.0, 0.0, 0.2, 0.1],
        [0.1, 1.0, 0.2, 0.1],
        [0.2, 0.1, 1.0, 0.1],
        [0.1, 0.2, 0.1, 1.0],
    ])
    d4_points = np.vstack([center + rng.normal(0.0, 0.06, size=(12, 4)) for center in topic_centers])
    d4_query = topic_centers[2] + np.array([0.02, 0.01, -0.02, 0.00])
    d4_centroids = topic_centers.copy()
    d4_assignments = np.repeat(np.arange(4), 12)
    d5_points = np.vstack([
        rng.normal([0.0, 0.0], 0.15, size=(15, 2)),
        rng.normal([2.0, 2.0], 0.15, size=(15, 2)),
        rng.normal([2.35, 2.05], 0.15, size=(15, 2)),
        rng.normal([4.0, 0.0], 0.15, size=(15, 2)),
    ])
    d5_points[15] = np.array([2.1705, 2.0305])
    d5_query = np.array([2.17, 2.03])
    d5_centroids = np.array([
        [0.0, 0.0],
        [2.18, 2.04],
        [2.0, 2.0],
        [4.0, 0.0],
    ])
    d5_assignments = np.concatenate([
        np.full(15, 0),
        np.full(15, 2),
        np.full(15, 1),
        np.full(15, 3),
    ])
    return [
        {"name": "D1 lesson six points", "points": d1_points, "query": d1_query, "centroids": d1_centroids, "assignments": d1_assignments, "nprobe": 1},
        {"name": "D2 clean clusters", "points": d2_points, "query": d2_query, "centroids": d2_centroids, "assignments": d2_assignments, "nprobe": 1},
        {"name": "D3 overlap and ties", "points": d3_points, "query": d3_query, "centroids": d3_centroids, "assignments": d3_assignments, "nprobe": 2},
        {"name": "D4 20newsgroups-style SVD vectors", "points": d4_points, "query": d4_query, "centroids": d4_centroids, "assignments": d4_assignments, "nprobe": 1},
        {"name": "D5 sparse long-tail clusters", "points": d5_points, "query": d5_query, "centroids": d5_centroids, "assignments": d5_assignments, "nprobe": 1},
    ]


def hybrid_ladder():
    return [
        {"name": "D1 lesson four docs", "bm25": [0.80, 0.20, 0.60, 0.10], "dense": [0.10, 0.90, 0.50, 0.30], "rerank": [0.30, 0.95, 0.55, 0.20], "relevant": {1, 2}, "top_k": 3},
        {"name": "D2 clean lexical plus semantic", "bm25": [0.95, 0.15, 0.65, 0.10, 0.05], "dense": [0.40, 0.88, 0.72, 0.20, 0.10], "rerank": [0.70, 0.92, 0.80, 0.10, 0.05], "relevant": {0, 1, 2}, "top_k": 5},
        {"name": "D3 scale mismatch and synonyms", "bm25": [9.0, 2.0, 6.0, 1.0, 0.5, 0.2], "dense": [0.10, 0.92, 0.55, 0.35, 0.20, 0.10], "rerank": [0.30, 0.98, 0.62, 0.40, 0.15, 0.05], "relevant": {1, 2, 3}, "top_k": 5},
        {"name": "D4 20newsgroups-style query labels", "bm25": [0.75, 0.20, 0.55, 0.48, 0.10, 0.05, 0.35], "dense": [0.55, 0.82, 0.62, 0.57, 0.25, 0.20, 0.52], "rerank": [0.78, 0.85, 0.65, 0.60, 0.20, 0.10, 0.58], "relevant": {0, 1, 2, 3}, "top_k": 5},
        {"name": "D5 exact-id plus vague long tail", "bm25": [12.0, 1.0, 0.5, 0.4, 5.5, 0.2, 0.1, 0.1], "dense": [0.05, 0.90, 0.88, 0.80, 0.20, 0.65, 0.30, 0.10], "rerank": [0.25, 0.95, 0.92, 0.82, 0.35, 0.70, 0.20, 0.05], "relevant": {1, 2, 3, 5}, "top_k": 5},
    ]


def cross_bi_ladder():
    return [
        {"name": "D1 lesson vectors", "query": [1.0, 0.0], "docs": [[0.9, 0.1], [0.7, 0.7], [0.0, 1.0]], "cross": [0.62, 0.88, 0.10], "rels": [2, 3, 0], "cutoff": 3},
        {"name": "D2 clean semantic pairs", "query": [1.0, 0.0, 0.0], "docs": [[0.9, 0.1, 0.0], [0.8, 0.2, 0.1], [0.1, 0.9, 0.0], [0.0, 0.2, 0.8]], "cross": [0.82, 0.90, 0.20, 0.10], "rels": [2, 3, 0, 0], "cutoff": 3},
        {"name": "D3 hard negatives and ties", "query": [1.0, 0.0, 0.0], "docs": [[0.95, 0.05, 0.0], [0.90, 0.10, 0.0], [0.85, 0.15, 0.0], [0.3, 0.9, 0.0], [0.0, 0.1, 0.9]], "cross": [0.55, 0.96, 0.70, 0.15, 0.05], "rels": [1, 3, 2, 0, 0], "cutoff": 4},
        {"name": "D4 lexical pair scorer subset", "query": [0.8, 0.1, 0.1, 0.0], "docs": [[0.7, 0.2, 0.1, 0.0], [0.6, 0.3, 0.1, 0.0], [0.2, 0.7, 0.1, 0.0], [0.1, 0.1, 0.8, 0.0], [0.5, 0.1, 0.3, 0.1], [0.0, 0.0, 0.1, 0.9]], "cross": [0.80, 0.86, 0.30, 0.20, 0.72, 0.05], "rels": [2, 3, 0, 0, 2, 0], "cutoff": 5},
        {"name": "D5 best doc below small cutoff", "query": [1.0, 0.0], "docs": [[0.99, 0.01], [0.96, 0.03], [0.93, 0.04], [0.90, 0.05], [0.75, 0.66], [0.10, 0.99]], "cross": [0.50, 0.52, 0.55, 0.60, 0.99, 0.05], "rels": [1, 1, 1, 2, 3, 0], "cutoff": 3},
    ]


def eval_ladder():
    return [
        {"name": "D1 lesson ranked list", "rels": [1, 0, 1, 1, 0], "slice": "lesson"},
        {"name": "D2 clean multi-query labels", "rels": [1, 1, 1, 0, 0], "slice": "head"},
        {"name": "D3 ties and incomplete judgments", "rels": [0, 1, 1, 0, 1], "slice": "ambiguous"},
        {"name": "D4 query-label subset", "rels": [1, 0, 1, 0, 0], "slice": "medium"},
        {"name": "D5 rare-query long tail", "rels": [0, 0, 0, 1, 1], "slice": "rare"},
    ]


## D1 concept: fuse sparse and dense scores
A simple hybrid score is $$s_h(d)=\alpha s_b(d)+(1-\alpha)s_e(d)$$. With $\alpha=0.5$, the lesson scores must be $[0.450,0.550,0.550,0.200]$.

In [ ]:

bm25 = np.array([0.80, 0.20, 0.60, 0.10])
dense = np.array([0.10, 0.90, 0.50, 0.30])
hybrid = 0.5 * bm25 + 0.5 * dense
print(np.round(hybrid, 3))
assert np.allclose(np.round(hybrid, 3), [0.450, 0.550, 0.550, 0.200])


## Add a transparent re-ranker
A shortlist re-ranker spends pairwise compute only on candidates. With $s_r=[0.30,0.95,0.55,0.20]$ and $\lambda=0.4$, final scores become $[0.360,0.790,0.550,0.200]$.

In [ ]:

rerank = np.array([0.30, 0.95, 0.55, 0.20])
hybrid_scores, final_order, final_scores = hybrid_fuse_and_rerank(bm25, dense, rerank, alpha=0.5, lam=0.4, normalize=False, top_k=4)
print(np.round(final_scores, 3))
print(final_order)
assert np.allclose(np.round(final_scores, 3), [0.360, 0.790, 0.550, 0.200])
assert final_order == [1, 2, 0, 3]


## Dataset ladder D1→D5
The ladder includes the lesson scores, clean lexical/semantic evidence, scale mismatch, a 20newsgroups-style label subset, and exact-id plus vague long-tail queries.

In [ ]:

rungs = hybrid_ladder()
for rung in rungs:
    print(rung["name"], "docs", len(rung["bm25"]), "relevant", sorted(rung["relevant"]))
    print("bm25", np.round(rung["bm25"][:4], 3))
    print("dense", np.round(rung["dense"][:4], 3))


## Run the same method across D1→D5
Metric: candidate recall@5 before the re-ranker, because a later stage cannot recover missing documents.

In [ ]:

hybrid_results = []
for rung in rungs:
    hybrid_scores, order, final_scores = hybrid_fuse_and_rerank(rung["bm25"], rung["dense"], rung["rerank"], alpha=0.5, lam=0.4, normalize=True, top_k=rung["top_k"])
    top = set(order[:5])
    relevant = set(rung["relevant"])
    recall = len(top & relevant) / len(relevant)
    hybrid_results.append({"name": rung["name"], "order": order, "scores": hybrid_scores, "metric": recall})
for item in hybrid_results:
    print(f"{item['name']}: recall@5={item['metric']:.3f} order={item['order']}")


## Results visualization
Left: sparse, dense, and hybrid ranking panels per rung. Right: candidate recall@5 curve.

In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, rung, result in zip(axes[:5], rungs, hybrid_results):
    labels = np.arange(len(rung["bm25"]))
    width = 0.25
    ax.bar(labels - width, minmax(rung["bm25"]), width, label="sparse")
    ax.bar(labels, minmax(rung["dense"]), width, label="dense")
    ax.bar(labels + width, result["scores"], width, label="hybrid")
    ax.set_title(rung["name"].split()[0])
    ax.set_xticks(labels)
    ax.set_xticklabels([f"d{i + 1}" for i in labels], rotation=90)
axes[0].legend(fontsize=7)
axes[5].plot(range(1, 6), [item["metric"] for item in hybrid_results], marker="o")
axes[5].set_ylim(-0.05, 1.05)
axes[5].set_xticks(range(1, 6))
axes[5].set_title("candidate recall@5")
fig.tight_layout()


## Pitfall on D5: averaging raw scores
Raw BM25 can have a much larger numeric range than dense scores. Averaging raw scores makes sparse scale dominate, so we normalize before fusion.

In [ ]:

hard = rungs[-1]
raw_scores, raw_order, raw_final = hybrid_fuse_and_rerank(hard["bm25"], hard["dense"], hard["rerank"], normalize=False, top_k=5)
norm_scores, norm_order, norm_final = hybrid_fuse_and_rerank(hard["bm25"], hard["dense"], hard["rerank"], normalize=True, top_k=5)
raw_recall = len(set(raw_order[:5]) & set(hard["relevant"])) / len(hard["relevant"])
norm_recall = len(set(norm_order[:5]) & set(hard["relevant"])) / len(hard["relevant"])
print("raw order", raw_order, "recall", raw_recall)
print("normalized order", norm_order, "recall", norm_recall)


## Evaluate it + Practice
- Metric: candidate recall@5 before reranking, plus a sparse-only no-skill baseline.
- Sanity check: all relevant docs in the union of sparse and dense candidates should be reachable.
- Ablation: turn dense scores off and semantic-only relevant docs should disappear.
- Failure signal: final precision improves while candidate recall silently falls.

Practice: sweep `alpha` from 0 to 1 and plot candidate recall@5.

Practice: use z-score normalization instead of min-max and compare the D5 order.